# Heart Disease Prediction — EDA & Statistical Analysis
**CSE570 Skill Based Assignment**

Covers rubric Section 2(b) — EDA and Statistical Analysis (10 marks).


Prerequisite: run the first notebook so `heart_disease_clean.csv` exists.

## 1. Setup and Load Cleaned Data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

df = pd.read_csv('heart_disease_clean.csv')

num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

print(df.shape)
df.head()

## 2. Univariate Analysis

### 2.1 Descriptive statistics
Central tendency, spread, and shape (skewness/kurtosis). Skewness guides our test choice: roughly symmetric features → Welch's t-test; skewed features → Mann-Whitney U.

In [ ]:
desc = df[num_cols].describe().T
desc['skew'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurtosis()
desc.round(2)

In [ ]:
# Auto-generated reading of the skewness table (edit the wording in your report)
for c_ in num_cols:
    sk = df[c_].skew()
    tag = 'strongly skewed -> Mann-Whitney U' if abs(sk) > 1 else ('moderately skewed' if abs(sk) > 0.5 else 'roughly symmetric -> Welch t-test')
    print(f'{c_:9s} skew = {sk:6.2f}  ({tag})')

**Your interpretation:** summarise the output above in 2-3 sentences (which features are skewed, and what that means for test choice).

### 2.2 Distribution shape of numeric features


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.ravel()
for i, c in enumerate(num_cols):
    sns.histplot(df[c], kde=True, ax=axes[i], bins=20, color='steelblue')
    axes[i].set_title(f'{c} (skew={df[c].skew():.2f})')
axes[5].axis('off')
plt.tight_layout()
plt.savefig('eda_univariate_distributions.png', dpi=150)
plt.show()

## 3. Bivariate Analysis — Numeric Features vs. Target

### 3.1 Group-wise descriptive comparison


In [ ]:
group_stats = df.groupby('target')[num_cols].agg(['mean', 'median', 'std']).round(2)
group_stats

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, c in zip(axes, num_cols):
    sns.boxplot(x='target', y=c, data=df, ax=ax)
    ax.set_xticklabels(['No disease', 'Disease'])
    ax.set_title(c)
plt.tight_layout()
plt.savefig('eda_numeric_by_target_boxplots.png', dpi=150)
plt.show()

In [ ]:
# Auto-generated: direction and size of group differences (disease vs no disease)
means = df.groupby('target')[num_cols].mean()
for c_ in num_cols:
    diff = means.loc[1, c_] - means.loc[0, c_]
    pct = diff / means.loc[0, c_] * 100
    direction = 'higher' if diff > 0 else 'lower'
    print(f'{c_:9s}: disease group mean is {direction} by {abs(diff):.2f} ({abs(pct):.1f}%)')

**Your interpretation:** link these differences to the clinical story (e.g. lower thalach = reduced exercise capacity) in 2-3 sentences.

### 3.2 Statistical tests — numeric features

**Test choice (justify in the report):**
- Welch's t-test for approximately normal features (age, trestbps, chol, thalach) — chosen over Student's t because it does not assume equal variances
- Mann-Whitney U for skewed features (oldpeak)

Reported: p-value, Cohen's d effect size, and 95% CI for the mean difference (confidence-based analysis). Alpha = 0.05.

In [ ]:
def welch_ttest(x1, x2):
    n1, n2 = len(x1), len(x2)
    m1, m2 = x1.mean(), x2.mean()
    v1, v2 = x1.var(ddof=1), x2.var(ddof=1)
    se = np.sqrt(v1/n1 + v2/n2)
    t_stat, p = stats.ttest_ind(x1, x2, equal_var=False)
    dfw = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
    tcrit = stats.t.ppf(0.975, dfw)
    ci = (m1 - m2 - tcrit*se, m1 - m2 + tcrit*se)
    sp = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
    d = (m1 - m2) / sp
    return t_stat, p, d, ci

results = []
for c in num_cols:
    g0, g1 = df.loc[df.target == 0, c], df.loc[df.target == 1, c]
    if abs(df[c].skew()) > 1:   # skewed -> non-parametric
        u, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        results.append([c, 'Mann-Whitney U', u, p, np.nan, np.nan])
    else:
        t, p, d, ci = welch_ttest(g0, g1)
        results.append([c, 'Welch t-test', t, p, d, f'({ci[0]:.2f}, {ci[1]:.2f})'])

res_df = pd.DataFrame(results, columns=['feature', 'test', 'statistic', 'p_value', 'cohens_d', '95% CI (mean diff)'])
res_df['significant (p<0.05)'] = res_df['p_value'] < 0.05
res_df.round(4)

In [ ]:
# Auto-generated summary of the numeric tests
sig = res_df[res_df['significant (p<0.05)']]
non = res_df[~res_df['significant (p<0.05)']]
print('Significant (p<0.05):', ', '.join(sig['feature']) or 'none')
print('Not significant     :', ', '.join(non['feature']) or 'none')
for _, r in sig.iterrows():
    if pd.notna(r['cohens_d']):
        size = 'large' if abs(r['cohens_d']) >= 0.8 else 'medium' if abs(r['cohens_d']) >= 0.5 else 'small'
        print(f"  {r['feature']}: Cohen's d = {r['cohens_d']:.2f} ({size}), 95% CI of mean diff {r['95% CI (mean diff)']}")

**Your interpretation:** a 95% CI that excludes 0 confirms the difference is statistically reliable. Cite this as your confidence-based analysis, and explain any non-significant feature (e.g. chol).

## 4. Bivariate Analysis — Categorical Features vs. Target

### 4.1 Chi-square test of independence
For each categorical feature we test H0: feature and target are independent, with Cramér's V as the effect size.

In [ ]:
def cramers_v(table):
    chi2 = stats.chi2_contingency(table)[0]
    n = table.values.sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k))

chi_results = []
for c in cat_cols:
    table = pd.crosstab(df[c], df['target'])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    chi_results.append([c, chi2, p, dof, cramers_v(table)])

chi_df = pd.DataFrame(chi_results, columns=['feature', 'chi2', 'p_value', 'dof', 'cramers_v'])
chi_df['significant (p<0.05)'] = chi_df['p_value'] < 0.05
chi_df.round(4)

In [ ]:
# Auto-generated summary of the chi-square tests
def strength(v): return 'strong' if v >= 0.5 else 'moderate' if v >= 0.3 else 'weak'
for _, r in chi_df.sort_values('cramers_v', ascending=False).iterrows():
    sig = 'significant' if r['p_value'] < 0.05 else 'NOT significant'
    print(f"{r['feature']:8s} chi2={r['chi2']:7.2f}  p={r['p_value']:.4f}  V={r['cramers_v']:.2f} ({strength(r['cramers_v'])}) -> {sig}")

**Your interpretation:** name the strongest associations and any non-significant feature (e.g. fbs), and give a short clinical reason.

### 4.2 Visual support for the key associations


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), cat_cols):
    ct = pd.crosstab(df[c], df['target'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#9ecae1', '#de2d26'])
    v = chi_df.loc[chi_df.feature == c, 'cramers_v'].values[0]
    ax.set_title(f'{c} (V={v:.2f})')
    ax.legend(['No disease', 'Disease'])
    ax.set_ylabel('Proportion')
plt.tight_layout()
plt.savefig('eda_categorical_proportions.png', dpi=150)
plt.show()

## 5. Multivariate Analysis

### 5.1 Full correlation matrix
Previews which features matter before modeling and motivates the feature-importance analysis later.

In [ ]:
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)
corr_with_target = df_enc.corr(numeric_only=True)['target'].drop('target').sort_values(key=abs, ascending=False)

print('Top correlations with target:')
print(corr_with_target.head(10).round(3))

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_enc.corr(numeric_only=True), cmap='coolwarm', center=0, annot=False)
plt.tight_layout()
plt.savefig('eda_full_correlation_heatmap.png', dpi=150)
plt.show()

### 5.2 Pairplot of the most target-correlated features


In [ ]:
top_feats = ['thalach', 'oldpeak', 'cp', 'ca', 'thal']
sns.pairplot(df[top_feats + ['target']], hue='target', diag_kind='kde', corner=True,
             palette={0: '#9ecae1', 1: '#de2d26'})
plt.tight_layout()
plt.savefig('eda_pairplot.png', dpi=150)
plt.show()